In [1]:
import pandas as pd

MasterDataFrame = pd.read_csv('CleanedMasterDataFrame.csv')
DropColumn = list(MasterDataFrame.columns)
MasterDataFrame = MasterDataFrame.drop(columns=DropColumn[0])

In [2]:
FeatureColumns = ['SI.POV.GINI','NY.GDP.MKTP.KD.ZG','SL.UEM.1524.ZS','FP.CPI.TOTL.ZG','GC.XPN.TOTL.GD.ZS']
GroupedByCountry = [group for _,group in MasterDataFrame.groupby('countryiso3code')]
for i in GroupedByCountry:
    CountryGroup = i

    for j in FeatureColumns:
        Lag1Name = j + '_Lag-1'  
        Lag2Name = j + '_Lag-2'
        Lag3Name = j + '_Lag-3'

        CountryGroup[Lag1Name] = CountryGroup[j].shift(periods=1,fill_value=pd.NA)
        CountryGroup[Lag2Name] = CountryGroup[j].shift(periods=2,fill_value=pd.NA)
        CountryGroup[Lag3Name] = CountryGroup[j].shift(periods=3,fill_value=pd.NA)


In [3]:
for i in GroupedByCountry: #iterating through the countries
    CountryGroup = i

    for j in FeatureColumns:
        NewColumnName = "DifferenceIn_" + j
        CountryGroup[NewColumnName] = CountryGroup[j].diff(periods=1)

MasterDataFrame = pd.concat(GroupedByCountry,axis=0,join='outer')

In [4]:
ListOfColumns = list(MasterDataFrame.columns)
for i in FeatureColumns:
    LagColumn1 = i + '_Lag-1'
    LagColumn2 = i + '_Lag-2'
    LagColumn3 = i + '_Lag-3' #have the names of the lag columns
    DifferenceColumn = 'DifferenceIn_' + i #have the name of the difference column
    ColumnsToBeMoved = [LagColumn1,LagColumn2,LagColumn3,DifferenceColumn]
    IndexOfIndicator = ListOfColumns.index(i) #have the index of the indicator column to move next to
    
    for i,j in enumerate(ColumnsToBeMoved):
        DestinationIndex = i+1+IndexOfIndicator #i should be the index of the list (0,1,2,3)
        ListOfColumns.remove(j) #j should be iterating through ColumnsToBeMoved
        ListOfColumns.insert(DestinationIndex,j)
MasterDataFrame = MasterDataFrame[ListOfColumns]

In [5]:
MasterDataFrame.to_csv('FeatureEngineeredDataFrame.csv')